<a href="https://colab.research.google.com/github/isabellamattar/FIAP-Data-Science/blob/main/titanic_knn_py.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Predição de Sobreviventes do Titanic com KNN

Isabella Simão Mattar - RM562481

**Objetivo:** Construir um modelo preditivo utilizando o algoritmo K-Nearest Neighbors (KNN) para estimar a sobrevivência dos passageiros do Titanic com base em suas características.

###Etapas Realizadas:
1. **Carregamento dos dados:** Importamos a base train.csv e selecionamos as features **(classe_social, sexo, idade, parentes, dependentes, tarifa, embarque)**.
2. **Limpeza e Preparação:**
   **Valores Nulos:** Preenchemos as idades ausentes com a **mediana** (para não distorcer a distribuição) e os embarques vazios com a **moda** (porto mais frequente).
   **Variáveis Categóricas:** Transformamos sexo em valores binários (0 e 1) e aplicamos o método One-Hot Encoding (get_dummies) no porto de embarque para transformar os textos em colunas numéricas.

3. **Separação Treino e Teste:** Dividimos a base usando train_test_split (70% para o modelo aprender, 30% para testar).

4. **Padronização (Crucial para o KNN):** Aplicamos o StandardScaler. Como o KNN calcula a "distância" matemática entre os passageiros, variáveis com números grandes (como a Tarifa) ofuscariam variáveis menores (como o Sexo). A padronização deixa todos na mesma escala.

5. **Treinamento e Avaliação:** Treinamos o KNeighborsClassifier com K=5 vizinhos e avaliamos o desempenho na base de teste.


###Conclusão dos Resultados:
O modelo atingiu uma **Acurácia de 79.10%**. Ele se mostrou mais eficiente em prever a classe majoritária (passageiros que não sobreviveram), atingindo um recall de 88% para essa classe, refletindo o desbalanceamento histórico dos dados da tragédia.

In [3]:
import pandas as pd
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from pathlib import Path

def carregar_dados(caminho_arquivo: str) -> pd.DataFrame:
    caminho = Path(caminho_arquivo)
    if not caminho.exists():
        raise FileNotFoundError(f"Arquivo não encontrado: {caminho_arquivo}. Fez o upload no Colab?")
    return pd.read_csv(caminho)

def pre_processar_dados(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.Series]:
    features = ['classe_social', 'sexo', 'idade', 'parentes', 'dependentes', 'tarifa', 'embarque']
    target = 'sobreviveu'

    X = df[features].copy()
    y = df[target].copy()

    X['idade'] = X['idade'].fillna(X['idade'].median())
    X['embarque'] = X['embarque'].fillna(X['embarque'].mode()[0])
    X['tarifa'] = X['tarifa'].fillna(X['tarifa'].median())

    X['sexo'] = X['sexo'].map({'male': 0, 'female': 1})
    X = pd.get_dummies(X, columns=['embarque'], drop_first=True)

    return X, y

def treinar_e_avaliar_modelo(X: pd.DataFrame, y: pd.Series, vizinhos: int = 5) -> None:
    """Separa treino/teste, padroniza a escala e treina o KNN."""
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    knn = KNeighborsClassifier(n_neighbors=vizinhos)
    knn.fit(X_train_scaled, y_train)

    y_pred = knn.predict(X_test_scaled)

    acuracia = accuracy_score(y_test, y_pred)
    relatorio = classification_report(y_test, y_pred)

    print(f"Acurácia do modelo (K={vizinhos}): {acuracia * 100:.2f}%")
    print(f"\nRelatório de Classificação:\n{relatorio}")

def main() -> None:
    """Função principal que organiza a bagunça toda."""
    arquivo_csv = 'train.csv'

    try:
        print("1. Iniciando o carregamento dos dados...")
        df = carregar_dados(arquivo_csv)

        print("2. Fazendo a limpeza e pré-processamento...")
        X, y = pre_processar_dados(df)

        print("3. Treinando o modelo KNN e avaliando resultados...\n")
        treinar_e_avaliar_modelo(X, y, vizinhos=5)

        print("\nSucesso! Pipeline finalizado.")

    except Exception as e:
        print(f"Deu ruim em alguma parte: {e}")

if __name__ == "__main__":
    main()

1. Iniciando o carregamento dos dados...
2. Fazendo a limpeza e pré-processamento...
3. Treinando o modelo KNN e avaliando resultados...

Acurácia do modelo (K=5): 79.10%

Relatório de Classificação:
              precision    recall  f1-score   support

           0       0.79      0.88      0.83       157
           1       0.80      0.67      0.73       111

    accuracy                           0.79       268
   macro avg       0.79      0.77      0.78       268
weighted avg       0.79      0.79      0.79       268


Sucesso! Pipeline finalizado.
